In [4]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
candidates.extend(parent / "PatchTrack" for parent in [cwd, *cwd.parents])
repo_root = next(
    (path for path in candidates if (path / "PatchTrack.py").is_file() and (path / "analyzer").is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError(f"Could not locate the PatchTrack project from {cwd}")
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

print(f"PatchTrack root: {repo_root}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
PatchTrack root: C:\theory_and_practice_of_software_design\PatchTrack


In [5]:
# No system libmagic dependency is required for classification.

In [6]:
from analyzer.main import PatchTrack
from analyzer import common
from analyzer import constant

In [7]:
# Classification reads only the extracted local patches; no GitHub token is needed.
data = []

In [8]:
print("Working directory:", os.getcwd())
print("Patch directory exists:", os.path.isdir("data/patches"))

Working directory: C:\theory_and_practice_of_software_design\PatchTrack
Patch directory exists: True


Create a magic cookie. This is needed during classification

In [9]:
from collections import Counter
import re

def _patch_tokens(path, root):
    path = Path(path)
    if not path.is_absolute():
        path = root / path

    text = path.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()

    # For unified diffs, compare added code rather than diff headers or deletions.
    is_diff = any(
        line.startswith(("diff --git ", "@@ ", "--- ", "+++ "))
        for line in lines
    )
    if is_diff:
        text = "\n".join(
            line[1:] for line in lines
            if line.startswith("+") and not line.startswith("+++")
        )

    return re.findall(r"[A-Za-z_]\w*|\d+(?:\.\d+)?|[^\s]", text)


def median_integration_rate(file_df, root=repo_root, classification="PA"):
    """Return per-PR ChatGPT-token containment rates and their median."""
    rows_by_pr = {}

    selected = file_df[file_df["File Classification"].eq(classification)]
    for _, row in selected.iterrows():
        key = (str(row["Pull Request"]), str(row["Repo"]))
        chatgpt_tokens = _patch_tokens(row["File Path"], root)
        github_tokens = _patch_tokens(row["GitHub Patch Path"], root)

        totals = rows_by_pr.setdefault(key, [Counter(), Counter()])
        totals[0].update(chatgpt_tokens)
        totals[1].update(github_tokens)

    rates = []
    for (pr, repo), (chatgpt_counts, github_counts) in rows_by_pr.items():
        denominator = sum(chatgpt_counts.values())
        if denominator:
            overlap = sum((chatgpt_counts & github_counts).values())
            rates.append({
                "Pull Request": pr,
                "Repo": repo,
                "Integration Rate (%)": 100 * overlap / denominator,
            })

    if not rates:
        raise ValueError(f"No usable {classification} pull requests found")

    rates.sort(key=lambda item: item["Integration Rate (%)"])
    middle = len(rates) // 2
    values = [item["Integration Rate (%)"] for item in rates]
    median = (
        values[middle]
        if len(values) % 2
        else (values[middle - 1] + values[middle]) / 2
    )
    return rates, median


In [10]:
common.verbose_mode = False

In [11]:
data = []

In [12]:
experiment = PatchTrack([])

In [13]:
pr_project_pair_records = experiment.build_pr_project_pairs()
pr_project_pair = [
    (pr, project)
    for pair in pr_project_pair_records
    for pr, project in pair.items()
]
print(f"PR/repository pairs: {len(pr_project_pair)}")

PR/repository pairs: 285


In [14]:
# take a peek of the the data
pr_project_pair[0:4]

[('79', '23-OSSCA-python-mysql-replication/python-mysql-replication'),
 ('214', '2DegreesInvesting/tiltIndicatorAfter'),
 ('13', '40LF0/couple_project_backend'),
 ('50', '4dn-dcic/foursight-core')]

In [15]:

print(len(pr_project_pair))

285


In [16]:
print("First five PR/repository pairs:", pr_project_pair[:5])

First five PR/repository pairs: [('79', '23-OSSCA-python-mysql-replication/python-mysql-replication'), ('214', '2DegreesInvesting/tiltIndicatorAfter'), ('13', '40LF0/couple_project_backend'), ('50', '4dn-dcic/foursight-core'), ('14', 'AdamAkiva/dashboard')]


In [17]:
# Classify the local patch set and build the DataFrames used below.
experiment.classify(pr_project_pair[-80:])
experiment.create_dataframes()
patch_classes = experiment.get_df_patch_classes()
classification_cache_version = 2

In [18]:
# Run once after classification to save it. In a later kernel, run setup cells,
# skip the expensive classify cell, then run this cell to restore the results.
import pickle

CLASSIFICATION_CACHE_VERSION = 2
classification_cache = repo_root / "output" / "patchtrack_classification_cache.pkl"
classification_cache.parent.mkdir(parents=True, exist_ok=True)

if experiment.df_patch_classes is not None:
    if globals().get("classification_cache_version") != CLASSIFICATION_CACHE_VERSION:
        raise RuntimeError(
            "Classification results were not recomputed with the current "
            "classifier. Run the classify cell before saving this cache."
        )
    classification_state = {
        "cache_version": CLASSIFICATION_CACHE_VERSION,
        "result_dict": experiment.result_dict,
        "pr_classifications": experiment.pr_classifications,
        "df_files_classes": experiment.df_files_classes,
        "df_patch_classes": experiment.df_patch_classes,
        "df_patches": experiment.df_patches,
    }
    with classification_cache.open("wb") as cache_file:
        pickle.dump(classification_state, cache_file)
    print(f"Saved classification cache to {classification_cache}")
else:
    if not classification_cache.is_file():
        raise FileNotFoundError(
            f"No classification cache found at {classification_cache}. "
            "Run the classify cell first to create it."
        )
    with classification_cache.open("rb") as cache_file:
        classification_state = pickle.load(cache_file)
    if classification_state.get("cache_version") != CLASSIFICATION_CACHE_VERSION:
        raise RuntimeError(
            "The classification cache is outdated and predates the corrected "
            "n-gram matching. Run the classify cell, then run this cell to "
            "create a new cache."
        )
    experiment.result_dict = classification_state["result_dict"]
    experiment.pr_classifications = classification_state["pr_classifications"]
    experiment.df_files_classes = classification_state["df_files_classes"]
    experiment.df_patch_classes = classification_state["df_patch_classes"]
    experiment.df_patches = classification_state["df_patches"]
    print(f"Loaded classification cache from {classification_cache}")

patch_classes = experiment.get_df_patch_classes()

Saved classification cache to C:\theory_and_practice_of_software_design\PatchTrack\output\patchtrack_classification_cache.pkl


In [19]:
print("Number of PR/project pairs:", len(pr_project_pair[:5]))
print("First 5 pairs:", pr_project_pair[:5])
print("Patch directory:", experiment.repo_dir_files)

Number of PR/project pairs: 5
First 5 pairs: [('79', '23-OSSCA-python-mysql-replication/python-mysql-replication'), ('214', '2DegreesInvesting/tiltIndicatorAfter'), ('13', '40LF0/couple_project_backend'), ('50', '4dn-dcic/foursight-core'), ('14', 'AdamAkiva/dashboard')]
Patch directory: data/patches/


In [20]:
patch_classes = experiment.get_df_patch_classes()
if patch_classes is None:
    raise RuntimeError("Classification did not produce patch-classification data")
Path("output").mkdir(exist_ok=True)

PA = patch_classes[patch_classes['Patch Classification'] == "PA"]
PA = PA.drop_duplicates(subset=['Pull Request', 'Repo'])
PA.to_csv(f'output/N{constant.NGRAM_SIZE}_PA_PATCH.csv', index=False)
PA.shape

(36, 7)

In [21]:
NE = patch_classes[patch_classes['Patch Classification'] == 'NE']
NE = NE.drop_duplicates(subset=['Pull Request', 'Repo'])
NE.to_csv(f'output/N{constant.NGRAM_SIZE}_NE_PATCH.csv', index=False)
NE.shape

(34, 7)

In [22]:
patch_classes = experiment.get_df_patch_classes()
PN = patch_classes[patch_classes['Patch Classification'] == 'PN']
PN = PN.drop_duplicates(subset=['Pull Request', 'Repo'])
PN.to_csv(f'output/N{constant.NGRAM_SIZE}_PN_PATCH.csv', index=False)
PN.shape

(10, 7)

In [23]:
d = experiment.get_df_file_classes()
if d is None:
    raise RuntimeError("Classification did not produce file-classification data")
d.to_csv(f'output/N{constant.NGRAM_SIZE}_ALL_PATCH_WITH_DUPLICATES.csv', index=False)
d.shape

(1924, 13)

In [24]:
df = experiment.get_df_file_classes()
if df is None:
    raise RuntimeError("Classification did not produce file-classification data")
df.drop_duplicates(subset=['Pull Request', 'Repo'], inplace=True)
df.to_csv(f'output/N{constant.NGRAM_SIZE}_ALL_PATCH.csv', index=False)
df.shape

(80, 13)

In [25]:
file_classes = experiment.get_df_file_classes()
print(file_classes["File Classification"].value_counts(dropna=False))
display(
    file_classes.loc[
        file_classes["File Classification"].eq("PA"),
        ["Pull Request", "Repo", "File Path", "GitHub Patch Path", "Similarity (%)"]
    ]
)

File Classification
PA              36
NOT EXISTING    34
PN              10
Name: count, dtype: int64


,Pull Request,Repo,File Path,GitHub Patch Path,Similarity (%)
119,931,SSWConsulting/SSW.Website,data/patches/SSWConsulting/SSW.Website\931\cha...,data/patches/SSWConsulting/SSW.Website\931\git...,54.55
932,8,timjuic/paypro-android,data/patches/timjuic/paypro-android\8\chatgpt\...,data/patches/timjuic/paypro-android\8\github\p...,27.45
130,766,starknet-id/app.starknet.id,data/patches/starknet-id/app.starknet.id\766\c...,data/patches/starknet-id/app.starknet.id\766\g...,87.50
1533,716,VOICEVOX/voicevox_engine,data/patches/VOICEVOX/voicevox_engine\716\chat...,data/patches/VOICEVOX/voicevox_engine\716\gith...,6.90
243,678,swarmion/swarmion,data/patches/swarmion/swarmion\678\chatgpt\pat...,data/patches/swarmion/swarmion\678\github\patc...,100.00
1524,655,VOICEVOX/voicevox_core,data/patches/VOICEVOX/voicevox_core\655\chatgp...,data/patches/VOICEVOX/voicevox_core\655\github...,20.00
1550,64,wahanegi/vibereport,data/patches/wahanegi/vibereport\64\chatgpt\pa...,data/patches/wahanegi/vibereport\64\github\pat...,8.33
1917,54172,WordPress/gutenberg,data/patches/WordPress/gutenberg\54172\chatgpt...,data/patches/WordPress/gutenberg\54172\github\...,2.63
1430,52,viets-software-club/truffle-ai-backend,data/patches/viets-software-club/truffle-ai-ba...,data/patches/viets-software-club/truffle-ai-ba...,12.90
1395,47,UNLV-CS472-672/2024-S-GROUP5-Munch,data/patches/UNLV-CS472-672/2024-S-GROUP5-Munc...,data/patches/UNLV-CS472-672/2024-S-GROUP5-Munc...,15.62


In [26]:
integration_rates, median_rate = median_integration_rate(file_classes)
print(f"Median PA integration rate: {median_rate:.2f}%")
display(integration_rates)

Median PA integration rate: 33.33%


[{'Pull Request': '54172',
  'Repo': 'WordPress/gutenberg',
  'Integration Rate (%)': 3.676470588235294},
 {'Pull Request': '1844',
  'Repo': 'VOICEVOX/voicevox',
  'Integration Rate (%)': 3.697478991596639},
 {'Pull Request': '291',
  'Repo': 'whatwg/html-build',
  'Integration Rate (%)': 5.381165919282512},
 {'Pull Request': '655',
  'Repo': 'VOICEVOX/voicevox_core',
  'Integration Rate (%)': 12.903225806451612},
 {'Pull Request': '1384',
  'Repo': 'VOICEVOX/voicevox',
  'Integration Rate (%)': 14.675767918088738},
 {'Pull Request': '116',
  'Repo': 'UNLV-CS472-672/2024-S-GROUP1-Roadwatch',
  'Integration Rate (%)': 15.141242937853107},
 {'Pull Request': '47',
  'Repo': 'UNLV-CS472-672/2024-S-GROUP5-Munch',
  'Integration Rate (%)': 16.129032258064516},
 {'Pull Request': '1836',
  'Repo': 'VOICEVOX/voicevox',
  'Integration Rate (%)': 18.27956989247312},
 {'Pull Request': '107',
  'Repo': 'UNLV-CS472-672/2024-S-GROUP1-Roadwatch',
  'Integration Rate (%)': 19.689119170984455},
 {'Pull

In [27]:
import plotly.graph_objects as go

fig = go.Figure(go.Box(
    y=[item["Integration Rate (%)"] for item in integration_rates],
    name="PA adoption rate",
    boxpoints="all",
    jitter=0.3,
    pointpos=0,
))
fig.add_hline(
    y=median_rate,
    line_dash="dash",
    annotation_text=f"Median: {median_rate:.2f}%",
)
fig.update_layout(
    title="PA Adoption Rate by Pull Request",
    yaxis_title="Adoption rate (%)",
    yaxis_range=[0, 100],
    showlegend=False,
)
fig.show()

In [28]:
unique_pr_classes = patch_classes.drop_duplicates(
    subset=["Pull Request", "Repo"]
)
unique_pr_classes["Patch Classification"].value_counts()

Patch Classification
PA    36
NE    34
PN    10
Name: count, dtype: int64

In [29]:
class_counts = unique_pr_classes["Patch Classification"].value_counts()

import plotly.graph_objects as go

fig = go.Figure(go.Bar(
    x=class_counts.index,
    y=class_counts.values,
    marker_color="#082a54"
))
fig.update_layout(
    xaxis_title="Patch Classification",
    yaxis_title="Number of Pull Requests",
    plot_bgcolor="white",
    font_color="black"
)
fig.show()

In [30]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Bar(
    x=['PA'],
    y=[int(patch_classes['Patch Classification'].eq('PA').sum())],
    name="Patch Applied (PA)",
    marker_color='#00A0A0'
))
fig.add_trace(go.Bar(
    x=['PN'],
    y=[int(patch_classes['Patch Classification'].eq('PN').sum())],
    name="Patch Not Applied (PN)",
    marker_color='#082a54'
))
fig.add_trace(go.Bar(
    x=['NE'],
    y=[int(patch_classes['Patch Classification'].eq('NE').sum())],
    name="Non-Existent (NE)",
    marker_color='#a00000'
))
fig.add_trace(go.Bar(
    x=['CC'],
    y=[int(patch_classes['Patch Classification'].eq('CC').sum())],
    name="Cannot Classify",
    marker_color='purple'
))
fig.add_trace(go.Bar(
    x=['ERROR'],
    y=[int(patch_classes['Patch Classification'].eq('ERROR').sum())],
    name="Error",
    marker_color='#A05000'
))
fig.update_layout(
    xaxis_title="Classifications",
    yaxis_title="Frequency",
    plot_bgcolor= "rgba(0, 0, 0, 0)",
    font_color="black",
    font_size=12,
    height=350,
    width=650,
    bargap=0.45
)
fig.update_xaxes(
    mirror=True,
    ticks='outside',
    # showline=True,
    linecolor='black',
    # gridcolor='lightgrey'
    # orientation='h'
)
fig.update_yaxes(
    mirror=True,
    ticks='outside',
    # showline=True,
    linecolor='black',
    gridcolor='lightgrey',
    # orientation='h'
    # tickangle=330
)
fig.update_layout(
    # autosize=True,
    # try to play around height and width values 
    legend=dict(
    yanchor="top",
    y=0.95,
    xanchor="left",
    x=0.61,
    bordercolor="gray",
    borderwidth=1,
    itemsizing="constant", 
    itemwidth=30,
    font=dict(
            # family="Courier",
            size=12,
        ),
))
fig.show()
# fig.write_image("classification.pdf", width=650, height=350)


In [31]:
unique_prs = patch_classes.drop_duplicates(subset=["Pull Request", "Repo"])
unique_prs["Patch Classification"].value_counts()

Patch Classification
PA    36
NE    34
PN    10
Name: count, dtype: int64

In [32]:
# Create the bar chart and modify as per the instructions
fig = go.Figure(go.Bar(
            x = [6, 9, 9, 10, 11, 19],
            y = ['Internal Improvements and Documentation',
                 'Clarification and Correction',
                 'Technical Limitations and Preferences',
                 'Specific Functionality Enhancement',
                 'Methodological Guidance',
                 'Adaptation and Tailored Solutions'
                 ],
            orientation='h',
            marker_color='#082a54'))

# Remove gray background and set background to white
fig.update_xaxes(
    mirror=True,
    ticks='outside',
    # showline=True,
    linecolor='black',
    # gridcolor='lightgrey'
    # orientation='h'
)
fig.update_yaxes(
    mirror=True,
    ticks='outside',
    # showline=True,
    linecolor='black',
    gridcolor='lightgrey',
    # orientation='h'
    # tickangle=330
)

# Add axis labels
fig.update_layout(
    # xaxis_title="Number of Animals",  # Label for X-axis
    # yaxis_title="Types of Animals",  # Label for Y-axis
    xaxis_title="Frequency",
    yaxis_title="Theme",
    plot_bgcolor= "rgba(0, 0, 0, 0)",
    font_color="black",
    font_size=12,
    height=300,
    width=600,
    # bargap=0.45
)

# Show the figure
fig.show()
# fig.write_image("pn_themes.pdf", width=600, height=300)

In [33]:
# Create the bar chart and modify as per the instructions
fig = go.Figure(go.Bar(
            x = [9, 11, 25, 61],
            y = [ 'Debugging and Optimization Strategies',
                 'Education and Knowledge Sharing',
                 'Documentation, Communication, & Translation',
                 'Conceptual Guidance & Theoretical Advice',
                 ],
            orientation='h',
            marker_color='#a00000'))

# Remove gray background and set background to white
fig.update_xaxes(
    mirror=True,
    ticks='outside',
    # showline=True,
    linecolor='black',
    # gridcolor='lightgrey'
    # orientation='h'
)
fig.update_yaxes(
    mirror=True,
    ticks='outside',
    # showline=True,
    linecolor='black',
    gridcolor='lightgrey',
    # orientation='h'
    # tickangle=330
)

# Add axis labels
fig.update_layout(
    # xaxis_title="Number of Animals",  # Label for X-axis
    # yaxis_title="Types of Animals",  # Label for Y-axis
    xaxis_title="Frequency",
    yaxis_title="Theme",
    plot_bgcolor= "rgba(0, 0, 0, 0)",
    font_color="black",
    font_size=12,
    height=300,
    width=600,
    # bargap=0.45
)

# Show the figure
fig.show()
# Export with fig.write_image(...) if Kaleido is installed.